# Day 060 — Exercise 5: HTTP ETag Caching

**ETags** are HTTP-level caching: the server includes an `ETag` header identifying the current version of the resource. The browser stores it and sends it back as `If-None-Match` on the next request. If the data hasn't changed, the server returns `304 Not Modified` (no body) — saving bandwidth.

This is different from in-process caching: ETags let the **client** avoid re-downloading data it already has.

In [ ]:
import hashlib
import json as _json
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse, Response
from starlette.testclient import TestClient


## Task

Implement `build_etag_api()` — return a FastAPI app with:

```
GET /data  → 200 JSON body + ETag + Cache-Control
GET /data  (If-None-Match == current ETag) → 304 empty body
GET /data  (If-None-Match != current ETag) → 200 JSON body
```

ETag format: `'"' + hashlib.md5(json_bytes).hexdigest() + '"'` (quoted string)

Use `request.headers.get('if-none-match', '')` to read the client's ETag.
Return `Response(status_code=304)` for the not-modified case.

## Your Implementation

In [ ]:
def build_etag_api() -> FastAPI:
    """FastAPI app with HTTP ETag-based caching on GET /data.

    GET /data  (no If-None-Match header)
        → 200 JSON + 'ETag: "<md5hash>"' + 'Cache-Control: max-age=60'

    GET /data  (If-None-Match matches current ETag)
        → 304 Not Modified (empty body)

    GET /data  (If-None-Match does NOT match)
        → 200 JSON + ETag + Cache-Control

    The ETag is the MD5 hex digest of the JSON-encoded data, wrapped in quotes.
    """
    # TODO: build app, compute etag from data, check if-none-match header
    raise NotImplementedError


In [ ]:
def build_etag_api() -> FastAPI:
    app   = FastAPI()
    _data = {"message": "Hello from the cached API!", "version": 1}

    def _etag(data) -> str:
        raw = _json.dumps(data, sort_keys=True)
        return '"' + hashlib.md5(raw.encode()).hexdigest() + '"'

    @app.get("/data")
    def get_data(request: Request):
        etag = _etag(_data)
        if_none_match = request.headers.get("if-none-match", "")
        if if_none_match == etag:
            return Response(status_code=304)
        return JSONResponse(
            content=_data,
            headers={"ETag": etag, "Cache-Control": "max-age=60"},
        )

    return app


## Automated checks

In [ ]:
score, total = 0, 5
try:
    app    = build_etag_api()
    client = TestClient(app, raise_server_exceptions=False)

    # first request → 200
    r1 = client.get("/data")
    assert r1.status_code == 200, f"Expected 200, got {r1.status_code}"
    score += 1; print("\u2705 GET /data returns 200")

    # has ETag header
    assert "etag" in r1.headers, f"Missing ETag header: {dict(r1.headers)}"
    etag = r1.headers["etag"]
    assert etag.startswith('"') and etag.endswith('"'), f"ETag must be quoted: {etag}"
    score += 1; print("\u2705 response has quoted ETag header")

    # has Cache-Control header
    cc = r1.headers.get("cache-control", "")
    assert "max-age" in cc, f"Expected max-age in Cache-Control, got {cc!r}"
    score += 1; print("\u2705 response has Cache-Control: max-age header")

    # matching ETag → 304
    r2 = client.get("/data", headers={"if-none-match": etag})
    assert r2.status_code == 304, f"Expected 304, got {r2.status_code}"
    assert len(r2.content) == 0, "304 response must have empty body"
    score += 1; print("\u2705 matching If-None-Match → 304 empty body")

    # wrong ETag → 200
    r3 = client.get("/data", headers={"if-none-match": '"wrong-etag"'})
    assert r3.status_code == 200, f"Expected 200, got {r3.status_code}"
    score += 1; print("\u2705 non-matching If-None-Match → 200")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def build_etag_api() -> FastAPI:
    app   = FastAPI()
    _data = {"message": "Hello from the cached API!", "version": 1}

    def _etag(data) -> str:
        raw = _json.dumps(data, sort_keys=True)
        return '"' + hashlib.md5(raw.encode()).hexdigest() + '"'

    @app.get("/data")
    def get_data(request: Request):
        etag = _etag(_data)
        if_none_match = request.headers.get("if-none-match", "")
        if if_none_match == etag:
            return Response(status_code=304)
        return JSONResponse(
            content=_data,
            headers={"ETag": etag, "Cache-Control": "max-age=60"},
        )

    return app
```

**Why quote the ETag?** The HTTP spec requires ETag values to be double-quoted strings: `\"abc123\"`, not `abc123`. Browsers and proxies reject unquoted ETags. The MD5 of the JSON body is a stable hash — any change to the data produces a different ETag, triggering a fresh 200 response.

</details>